# Person mentions in TEI with an LLM on the DHinfra cluster

The same task as in notebook 1, on the same ten letters: mark every mention of a real person
(`<persName>` with its name parts) and every pronoun or phrase that refers to one (`<rs>`). This
time the model gets the TEI fragment of the letter and returns it with the markup added.

That is the obvious way to ask, and it has a cost: **the model writes the letter back itself.** It
can re-indent it, correct a spelling, drop or rewrite a footnote. The prompt forbids all of this;
the checks in section 5 look whether it happened anyway:

- **parse**: is the answer well-formed XML (`strict`), or does it only parse with error recovery
  (`recovered`), or not at all (`error`)?
- **text**: is the letter text `identical`, the same apart from spacing (`whitespace`), or
  `changed`?
- **notes**: are the footnotes `intact`, or `lost`, `added`, `changed`, `annotated`?

Section 6 compares the mentions with those of notebook 1, if you ran it with the same model. No
GPU is needed in your own session: the model runs on the cluster.

**Steps:** 1. set up, 2. the prompt, 3. one letter, 4. all ten letters, 5. check the results,
6. compare with notebook 1.

## 1. Set up

In [ ]:
# Run once per session. The cluster's Python environment may already have these.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

Your project token is read from `DHINFRA_KEY`, either from the environment or from a `.env`
file next to this notebook (`DHINFRA_KEY="..."`). Never commit the `.env` file.

In [ ]:
import json
from pathlib import Path
import concurrent.futures as cf

import pandas as pd

import helpers

MODEL = "deepseek-v4.1-flash"   # pick a name from the list printed below
THINKING = False                # True: the model reasons first (slower, more tokens)
WORKERS = 4                     # parallel requests; keep this small, the cluster is shared

PROMPT_DIR = Path("prompts/tei")
LETTER_DIR = Path("data/letters")
METADATA_FILE = Path("data/metadata.json")
OUTPUT_DIR = Path("output") / MODEL / "tei"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
POSITIONS_DIR = Path("output") / MODEL / "positions"   # results of notebook 1

client = helpers.get_client()
# Optional: list the models the API offers, to check that MODEL is spelled correctly.
print([m.id for m in client.models.list().data])

## 2. The prompt

The *user message* has the same metadata block as in notebook 1, followed by the TEI fragment as
it is in the edition: footnotes, line breaks and indentation included. The *system prompt* has
the same three parts: `prompt.txt`, `encoding_rules.txt` (which here also says: change nothing
but the markup, not even whitespace) and `few_shot_examples.txt`.

In [ ]:
metadata = json.loads(METADATA_FILE.read_text(encoding="utf-8"))

name = "letter2276"
div = helpers.read_letter(LETTER_DIR / f"{name}.xml")
print(helpers.tei_message(metadata[name], div))

In [ ]:
system_prompt = helpers.build_system_prompt(PROMPT_DIR)
print(f"{len(system_prompt):,} characters")
print(system_prompt[:2500], "...")

## 3. One letter

In [ ]:
raw, usage = helpers.ask_model(
    client, MODEL, system_prompt, helpers.tei_message(metadata[name], div), thinking=THINKING)
print(raw)
print()
print("tokens in/out:", usage.prompt_tokens, usage.completion_tokens)

To check the answer, the plain text is taken from the returned fragment exactly as notebook 1
takes it from the edition's file (footnotes out, blocks end a word, whitespace collapsed), and
the two are compared. Only if the text is identical, or differs in spacing alone, are the
mentions read: their positions are then carried over onto the original letter text.

In [ ]:
original = helpers.letter_text(div)
returned_div, parse = helpers.parse_fragment(raw)
print("parse:", parse)
if returned_div is not None:
    returned = helpers.letter_text(returned_div)
    print("text: ", helpers.compare_text(returned["text"], original["text"]))
    print("notes:", helpers.compare_notes(returned, original))
    for s in helpers.tei_to_spans(returned, original):
        parts = ", ".join(f"{p['element']}={p['text']!r}" for p in s["parts"])
        print(f"{s['element']:9} {s['text']!r:40} {parts}")

## 4. All ten letters

One request per letter, a few at a time. For each letter, `output/<model>/tei/` gets the returned
fragment as `.xml` and a `.json` file with the checks, the mentions and the token counts.

In [ ]:
def run_one(name):
    div = helpers.read_letter(LETTER_DIR / f"{name}.xml")
    raw, usage = helpers.ask_model(
        client, MODEL, system_prompt, helpers.tei_message(metadata[name], div), thinking=THINKING)
    (OUTPUT_DIR / f"{name}.xml").write_text(raw, encoding="utf-8")
    original = helpers.letter_text(div)
    returned_div, parse = helpers.parse_fragment(raw)
    result = {"parse": parse, "text": None, "notes": None, "spans": [], "difference": None,
              "tokens_in": usage.prompt_tokens, "tokens_out": usage.completion_tokens}
    if returned_div is not None:
        returned = helpers.letter_text(returned_div)
        result["text"] = helpers.compare_text(returned["text"], original["text"])
        result["notes"] = helpers.compare_notes(returned, original)
        if result["text"] == "changed":
            result["difference"] = helpers.first_difference(returned["text"], original["text"])
        else:
            result["spans"] = helpers.tei_to_spans(returned, original)
    (OUTPUT_DIR / f"{name}.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return original, result

results = {}
with cf.ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(run_one, n): n for n in metadata}
    for future in cf.as_completed(futures):
        n = futures[future]
        try:
            results[n] = future.result()
            print("done  ", n)
        except Exception as err:   # keep going; one failed letter should not stop the batch
            print("FAILED", n, "-", err)

## 5. Check the results

*parse*, *text* and *notes* as described at the top. A letter whose text came back `changed` is
not used further, as in the full pipeline: its mentions would sit on a text that is not the
letter. The columns *kept ... added* are the same comparison with the edition's markup as in
notebook 1.

In [ ]:
rows = []
for n, (original, result) in sorted(results.items()):
    usable = result["text"] in ("identical", "whitespace")
    rows.append({
        "letter": n,
        "language": metadata[n]["language"],
        "parse": result["parse"],
        "text": result["text"],
        "notes": result["notes"],
        "persName": sum(s["element"] == "persName" for s in result["spans"]) if usable else None,
        "rs": sum(s["element"] == "rs" for s in result["spans"]) if usable else None,
        **(helpers.compare_with_edition(original, result["spans"]) if usable else {}),
        "tokens_out": result["tokens_out"],
    })
table = pd.DataFrame(rows).set_index("letter")
table

In [ ]:
# Where the text of a letter came back changed: the first difference.
for n, (original, result) in sorted(results.items()):
    if result["difference"]:
        d = result["difference"]
        print(f"{n}, character {d['position']}:")
        print("  original:", repr(d["original"]))
        print("  returned:", repr(d["returned"]))

## 6. Compare with notebook 1

If `01_persons_by_position.ipynb` was run with the same `MODEL`, its results are in
`output/<model>/positions/`. For each letter, the mentions both ways found at exactly the same
place are counted, and those only one of them found. The last two columns compare the output
tokens: a list of word numbers against the whole fragment written out again.

In [ ]:
def keys(spans):
    return {(s["element"], s["start"], s["end"]) for s in spans}

rows = []
for n, (original, result) in sorted(results.items()):
    path = POSITIONS_DIR / f"{n}.json"
    if not path.exists() or result["text"] not in ("identical", "whitespace"):
        continue
    positions = json.loads(path.read_text(encoding="utf-8"))
    a, b = keys(positions["spans"]), keys(result["spans"])
    rows.append({"letter": n, "both": len(a & b), "only_positions": len(a - b), "only_tei": len(b - a),
                 "tokens_out_positions": positions["tokens_out"], "tokens_out_tei": result["tokens_out"]})

if rows:
    comparison = pd.DataFrame(rows).set_index("letter")
    display(comparison)
    print("output tokens, all letters: positions", comparison.tokens_out_positions.sum(),
          "- TEI", comparison.tokens_out_tei.sum())
else:
    print(f"Nothing to compare: run notebook 1 with MODEL = {MODEL!r} first.")

In [ ]:
# The mentions of one letter side by side. Change the name to see another one.
n = "letter1222"
original = results[n][0]
positions = json.loads((POSITIONS_DIR / f"{n}.json").read_text(encoding="utf-8")) if (POSITIONS_DIR / f"{n}.json").exists() else {"spans": []}
a = {(s["element"], s["start"], s["end"]): s["text"] for s in positions["spans"]}
b = {(s["element"], s["start"], s["end"]): s["text"] for s in results[n][1]["spans"]}
for key in sorted(a.keys() | b.keys(), key=lambda k: k[1]):
    where = "both" if key in a and key in b else ("positions" if key in a else "tei")
    print(f"{where:10} {key[0]:9} {(a.get(key) or b.get(key))!r}")

## Where next

- Change `MODEL` or `THINKING` and run both notebooks again. Which way of asking works better
  can differ from model to model.
- The full pipeline uses the TEI answer the same way as here: it reads the mentions from it and
  writes them into the edition's file itself, never the model's version of the letter.